# Build EarthCatalog from Scratch (fresh bulk ingest + delta)

Provisions a Coiled cluster, wipes the destination prefix, runs a full catalog
build against the ITS_LIVE S3 Inventory, then appends a delta from a newer
inventory snapshot.

**Steps:**
1. Set all parameters in the `Parameters` cell below
2. Safety check + delete the destination AWS prefix
3. Verify AWS credentials
4. Resolve the S3 Inventory manifests
5. Define `create_client` (Coiled cluster)
6. Run the **full** ingest via `scripts/run_backfill.run(mode="full")`
7. Inspect the catalog
8. Run the **delta** ingest (newer inventory) via `daily_delta` + `run(mode="delta")`
9. Shut down the cluster

> ⚠️ **Check `CATALOG_BASE` before running.**  The safety guard stops you if it
> matches the live catalog prefix.

## Parameters

In [ ]:
# ============================================================
# DESTINATION  — edit before running
# ============================================================

# All outputs (warehouse/, warehouse_index.parquet, delta/,
# earthcatalog.db) land under this prefix.
CATALOG_BASE = "s3://its-live-data/test-space/stac/scratch-build-02"

# Derived — no need to edit
WAREHOUSE    = f"{CATALOG_BASE.rstrip('/')}/warehouse"
INDEX_URI    = f"{CATALOG_BASE.rstrip('/')}/warehouse_index.parquet"
DELTA_PREFIX = f"{CATALOG_BASE.rstrip('/')}/delta"
LOCAL_CATALOG = "/tmp/earthcatalog_fresh.db"   # local SQLite path

# Catalog object key within the bucket (used for catalog upload)
_base_key   = CATALOG_BASE.removeprefix("s3://").split("/", 1)[1].rstrip("/")
CATALOG_KEY = f"{_base_key}/earthcatalog.db"
LOCK_KEY    = f"{_base_key}/.lock"

# ============================================================
# INVENTORY — predefined S3 Inventory destination prefix
# ============================================================

INVENTORY_BASE = (
    "s3://pds-buckets-its-live-logbucket-70tr3aw5f2op/inventory/"
    "velocity_image_pair/its-live-data/VelocityGranuleInventory"
)

# Snapshot date for the FULL build.  None = latest available.
# e.g. "2026-08-03"  (snapshot folders are named <date>T01-00Z)
FULL_INVENTORY_DATE = None

# Snapshot date for the DELTA build.  Must be NEWER than the full snapshot.
# None = latest available (warned if it equals the full snapshot).
DELTA_INVENTORY_DATE = None

# ============================================================
# GRID (fresh full builds)
# ============================================================

GRID_TYPE       = "h3"   # "h3" | "s2" | "utm" | "geojson"
GRID_RESOLUTION = 1      # h3=1 (~86k km²/cell); s2 default is 2

# ============================================================
# INGEST TUNING
# ============================================================

CHUNK_SIZE   = 100_000   # items per fetch chunk
COMPACT_ROWS = 100_000   # max rows per output GeoParquet file
LIMIT        = None      # set an int to smoke-test the first N items

# ============================================================
# COILED CLUSTER
# ============================================================

# Paste a scheduler address to reuse a running cluster and skip provisioning,
# e.g. "tls://scheduler-abc123.us-west-2.aws.dask.host:8786".  Leave None to
# provision a fresh cluster.
COILED_SCHEDULER_ADDRESS = None

COILED_N_WORKERS   = 8
COILED_VM_TYPE     = "c6i.2xlarge"
COILED_THREADS     = 2
COILED_CLUSTER_NAME = "earthcatalog-fresh-build"
COILED_REGION      = "us-west-2"
COILED_SPOT_POLICY = "spot_with_fallback"   # "spot" | "on_demand"

# ============================================================
# AWS
# ============================================================

AWS_REGION = "us-west-2"

# --- helpers + summary ---
def _parse_s3_uri(uri: str):
    rest = uri.removeprefix("s3://")
    bucket, _, key = rest.partition("/")
    return bucket, key

print("Parameters")
print(f"  Catalog base  : {CATALOG_BASE}")
print(f"  Warehouse     : {WAREHOUSE}")
print(f"  Index         : {INDEX_URI}")
print(f"  Delta prefix  : {DELTA_PREFIX}")
print(f"  Local DB      : {LOCAL_CATALOG}")
print(f"  Catalog key   : {CATALOG_KEY}")
print(f"  Inventory base: {INVENTORY_BASE}")
print(f"  Grid          : {GRID_TYPE} / resolution {GRID_RESOLUTION}")
print(f"  Workers       : {COILED_N_WORKERS}× {COILED_VM_TYPE}")
print(f"  Chunk size    : {CHUNK_SIZE:,}")
print(f"  Limit         : {LIMIT}")

## Safety check

In [ ]:
PROTECTED = "s3://its-live-data/test-space/stac/catalog"

if CATALOG_BASE.rstrip("/") == PROTECTED.rstrip("/"):
    raise ValueError(
        f"CATALOG_BASE is set to the production path ({PROTECTED}).\n"
        "Change it to a new prefix before proceeding."
    )

print(f"OK — {CATALOG_BASE!r} does not clobber {PROTECTED!r}")

## Delete the destination prefix

Removes any prior build (warehouse files, index, delta, catalog.db) under
`CATALOG_BASE` so the full build starts from zero.  Runs in **dry-run** mode
until you set `DELETE_DRY_RUN = False`.

In [ ]:
import obstore

from earthcatalog.inventory import get_authenticated_store

DELETE_DRY_RUN = True   # set False to actually delete

def delete_prefix(s3_uri: str, dry_run: bool = True) -> int:
    bucket, key = _parse_s3_uri(s3_uri)
    store = get_authenticated_store(bucket)
    prefix = key.rstrip("/") + "/"
    count = 0
    for batch in obstore.list(store, prefix=prefix):
        for obj in batch:
            count += 1
            if not dry_run:
                obstore.delete(store, obj["path"])
    return count

n = delete_prefix(CATALOG_BASE, dry_run=DELETE_DRY_RUN)
action = "Would delete" if DELETE_DRY_RUN else "Deleted"
print(f"{action} {n:,} object(s) under {CATALOG_BASE.rstrip('/')}/")
if DELETE_DRY_RUN:
    print("Set DELETE_DRY_RUN = False and re-run to actually delete.")

## Verify AWS credentials

In [ ]:
import boto3

identity = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()
print(f"Account : {identity['Account']}")
print(f"UserId  : {identity['UserId']}")
print(f"ARN     : {identity['Arn']}")

## Resolve inventory manifests

S3 Inventory writes dated snapshot folders under `INVENTORY_BASE`.  This cell
pins a date if one is given, otherwise discovers the latest `manifest.json`.

In [ ]:

def resolve_manifest(date_str=None) -> str:
    bucket, key = _parse_s3_uri(INVENTORY_BASE)
    if date_str:
        return f"{INVENTORY_BASE}/{date_str}T01-00Z/manifest.json"
    store = get_authenticated_store(bucket)
    found = []
    for batch in obstore.list(store, prefix=key.rstrip("/") + "/"):
        for obj in batch:
            p = obj["path"]
            if p.endswith("manifest.json"):
                found.append(p)
    found.sort(reverse=True)
    if not found:
        raise RuntimeError(f"No manifest.json found under {INVENTORY_BASE}")
    return f"s3://{bucket}/{found[0]}"

FULL_MANIFEST  = resolve_manifest(FULL_INVENTORY_DATE)
DELTA_MANIFEST = resolve_manifest(DELTA_INVENTORY_DATE)

print(f"Full  manifest : {FULL_MANIFEST}")
print(f"Delta manifest : {DELTA_MANIFEST}")
if FULL_MANIFEST == DELTA_MANIFEST:
    print("WARN: FULL and DELTA manifests are identical — the delta will add nothing.")
    print("      Pin FULL_INVENTORY_DATE to an earlier snapshot to observe a real delta.")

## Define `create_client` (Coiled cluster)

`run_backfill` calls `create_client()` once per ingest (full, then delta).  The
function is idempotent — the first call provisions the cluster, later calls
return the existing client.  AWS credentials are forwarded to workers so they
can read/write S3.

> Coiled ships the notebook's environment to the workers, so as long as this
> notebook runs from an environment with `earthcatalog` installed, the workers
> get the same packages automatically.

In [ ]:
import os

# Populated by create_client so the shutdown cell can close them.
cluster = None
client  = None

def create_client():
    """Return a Dask client, provisioning a Coiled cluster on first call."""
    global cluster, client

    if client is not None:
        return client

    if COILED_SCHEDULER_ADDRESS:
        from dask.distributed import Client as DaskClient
        print(f"Connecting to: {COILED_SCHEDULER_ADDRESS}")
        client = DaskClient(COILED_SCHEDULER_ADDRESS)
        print(f"Dashboard: {client.dashboard_link}")
        return client

    import coiled
    from dask.distributed import Client as DaskClient

    print(
        f"Provisioning '{COILED_CLUSTER_NAME}' — {COILED_N_WORKERS}× {COILED_VM_TYPE} "
        f"({COILED_SPOT_POLICY}) …"
    )
    cluster = coiled.Cluster(
        n_workers=COILED_N_WORKERS,
        worker_vm_types=[COILED_VM_TYPE],
        region=COILED_REGION,
        name=COILED_CLUSTER_NAME,
        worker_options={"nthreads": COILED_THREADS},
        spot_policy=COILED_SPOT_POLICY,
    )
    client = DaskClient(cluster)

    print(f"Dashboard        : {client.dashboard_link}")
    print(f"Scheduler address: {client.scheduler.address}")
    print("  (paste into COILED_SCHEDULER_ADDRESS to reuse without re-provisioning)")

    # Forward AWS credentials so workers can read/write S3.
    aws_envs = {
        k: os.environ[k]
        for k in (
            "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY",
            "AWS_SESSION_TOKEN",
            "AWS_DEFAULT_REGION",
        )
        if k in os.environ
    }
    if aws_envs:
        cluster.send_private_envs(aws_envs)
        print(f"AWS credentials forwarded to workers ({', '.join(aws_envs)}).")
    else:
        print("WARN: no AWS credentials found in environment — workers may lack S3 access.")

    return client

print("create_client defined.")

## Full ingest

Runs the resumable `bulk_ingest` / `Ingester` pipeline with `mode="full"`:
drops + recreates the Iceberg table, fetches STAC JSONs on the cluster, writes
GeoParquet, and uploads `earthcatalog.db`.  The unified index is the resume
checkpoint, so a failed run can simply be re-invoked.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from earthcatalog.config import GridConfig  # noqa: E402
from scripts.run_backfill import run as run_backfill  # noqa: E402

run_backfill(
    inventory=FULL_MANIFEST,
    catalog=LOCAL_CATALOG,
    warehouse=WAREHOUSE,
    catalog_key=CATALOG_KEY,
    lock_key=LOCK_KEY,
    mode="full",
    grid=GridConfig(type=GRID_TYPE, resolution=GRID_RESOLUTION),
    chunk_size=CHUNK_SIZE,
    compact_rows=COMPACT_ROWS,
    limit=LIMIT,
    create_client=create_client,
)

## Inspect the catalog

In [ ]:
import io
import subprocess
import sys

import pyarrow.parquet as pq

result = subprocess.run(
    [
        sys.executable, str(REPO_ROOT / "scripts" / "info.py"),
        "--catalog",   LOCAL_CATALOG,
        "--warehouse", WAREHOUSE,
    ],
    capture_output=True, text=True, cwd=str(REPO_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# Unified index size (footer read — no full scan)
_bucket, _key = _parse_s3_uri(INDEX_URI)
_store = get_authenticated_store(_bucket)
_meta = _store.head(_key)
_tail = bytes(obstore.get_range(_store, _key, start=_meta["size"] - 65536, length=65536))
_n_rows = pq.ParquetFile(io.BytesIO(_tail)).metadata.num_rows
print(f"Unified index rows : {_n_rows:,}")

## Delta ingest (newer inventory)

1. `run_daily_delta` streams the newer inventory, anti-joins it against the
   unified index, and writes `{DELTA_PREFIX}/pending/delta_<date>.parquet` with
   only the **new** keys.
2. `run_backfill(mode="delta")` consumes that delta and appends to the
   warehouse + index (the same cluster is reused).

> For a tiny delta you can drop the cluster: pass `create_client=None` and
> `scheduler="synchronous"` to step 2.

In [ ]:
from datetime import UTC, datetime

from scripts.daily_delta import run_daily_delta

DELTA_DATE = datetime.now(UTC).strftime("%Y-%m-%d")

delta_result = run_daily_delta(
    manifest_uri=DELTA_MANIFEST,
    warehouse_hash_uri=INDEX_URI,
    delta_prefix=DELTA_PREFIX,
    date_str=DELTA_DATE,
)

DELTA_INVENTORY = delta_result["delta_key"]
print(f"Delta new items : {delta_result['new_items']:,}")
print(f"Delta parquet   : {DELTA_INVENTORY}")

run_backfill(
    inventory=DELTA_INVENTORY,
    catalog=LOCAL_CATALOG,
    warehouse=WAREHOUSE,
    catalog_key=CATALOG_KEY,
    lock_key=LOCK_KEY,
    mode="delta",
    chunk_size=CHUNK_SIZE,
    compact_rows=COMPACT_ROWS,
    limit=LIMIT,
    create_client=create_client,
)

## Inspect after delta

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(REPO_ROOT / "scripts" / "info.py"),
        "--catalog",   LOCAL_CATALOG,
        "--warehouse", WAREHOUSE,
    ],
    capture_output=True, text=True, cwd=str(REPO_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## Shut down the cluster

In [ ]:
if client:
    client.close()
if cluster:
    cluster.close()
    print("Cluster shut down.")
else:
    print("Using pre-existing scheduler — nothing to shut down from this notebook.")